# Labels and split (BTXRD)
Gives every image its label (normal or tumor) and saves it as `labels.csv`. Then splits the images into `train`, `validation` and `test` folders inside `BTXRD no duplicates`, with one subfolder per label. The images in `images` are not changed.

In [1]:
import shutil
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split

BTXRD = Path("../data/BTXRD no duplicates")

## Labels
An image **with** an annotation file is a tumor image. An image **without** one is normal.

In [2]:
rows = []
for path in sorted(BTXRD.glob("images/*.jp*g")):  # .jpg and .jpeg
    annotation = BTXRD / "Annotations" / (path.stem + ".json")
    if annotation.exists():
        label = "tumor"
    else:
        label = "normal"
    rows.append({"file": path.name, "label": label})
labels = pd.DataFrame(rows)

print("Images:", len(labels))
print(labels["label"].value_counts().to_string())
labels.head()

Images: 3723
label
normal    1866
tumor     1857


,file,label
0,IMG000001.jpeg,tumor
1,IMG000002.jpeg,tumor
2,IMG000003.jpeg,tumor
3,IMG000004.jpeg,tumor
4,IMG000005.jpeg,tumor


In [3]:
labels.to_csv(BTXRD / "labels.csv", index=False)
print("Saved:", BTXRD / "labels.csv")

Saved: ../data/BTXRD no duplicates/labels.csv


## Split
70% train, 15% validation, 15% test. Each part keeps the same share of normal and tumor (`stratify`), and the fixed seed (`random_state=42`) gives the same split every time.

In [4]:
# first cut: 70% train, 30% rest. second cut: the rest in half
train, rest = train_test_split(labels, test_size=0.30, stratify=labels["label"], random_state=42)
validation, test = train_test_split(rest, test_size=0.50, stratify=rest["label"], random_state=42)

print("Train:", len(train), " Validation:", len(validation), " Test:", len(test))

Train: 2606  Validation: 558  Test: 559


In [5]:
# copy every image into split/label
for split, part in [("train", train), ("validation", validation), ("test", test)]:
    for file, label in zip(part["file"], part["label"]):
        folder = BTXRD / split / label
        folder.mkdir(parents=True, exist_ok=True)
        shutil.copy(BTXRD / "images" / file, folder / file)

In [6]:
for split in ["train", "validation", "test"]:
    print(split)
    for label in ["normal", "tumor"]:
        print("   ", label, len(list((BTXRD / split / label).glob("*.jp*g"))))

train
    normal 1306
    tumor 1300
validation
    normal 280
    tumor 278
test
    normal 280
    tumor 279
